In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import numpy as np
import matplotlib.tri as mtri
from scipy.spatial import Delaunay
import matplotlib.pyplot as plt
from astropy.table import Table
from astropy.io import ascii
import pandas as pd
import networkx as nx
import scipy
import seaborn as sns
from scipy.stats import norm
from mpl_toolkits.mplot3d import Axes3D
from sklearn.decomposition import PCA
import os
from collections import Counter
from concurrent.futures import ProcessPoolExecutor
from collections import defaultdict
import gc
import glob

In [3]:
def dict_data(n_random, numero, start_index=0):
    data = {}
    rand = {}
    for i in range(n_random):
        file_index = start_index + i
        table = Table.read(
            f"/content/drive/MyDrive/DESI/classification/LRG_NGC_{numero}_random_{file_index}_filt.fits"
        ).to_pandas()

        table['type'] = table['type'].apply(lambda x: x.decode() if isinstance(x, bytes) else x)
        table[f'class_{file_index}'] = table[f'class_{file_index}'].apply(
            lambda x: x.decode() if isinstance(x, bytes) else x
        )

        data[file_index] = dict(zip(table[table['type']=='data']['TARGETID'],
                                    table[table['type']=='data'][f'class_{file_index}']))
        rand[file_index] = dict(zip(table[table['type']=='rand']['TARGETID'],
                                    table[table['type']=='rand'][f'class_{file_index}']))
    return data, rand

In [4]:
def classes(data):
    dict_classes = {}
    for _, dic in data.items():
        for tid, clase in dic.items():
            dict_classes.setdefault(tid, []).append(clase)
    return dict_classes

In [5]:
def compute_class_fractions(class_dict,n_random):
    count_result = {}
    total = n_random
    for tid, class_list in class_dict.items():
        counts = Counter(class_list)
        p = np.array([
            counts.get('filament', 0) / total*100,
            counts.get('sheet', 0) / total*100,
            counts.get('void', 0) / total*100,
            counts.get('knot', 0) / total*100
        ])

        count_result[tid] = p

    return count_result

In [6]:
def cf_total(dict_data):
  filament = []
  sheet = []
  void = []
  knot = []
  for i in dict_data.values():
    filament.append(i[0])
    sheet.append(i[1])
    void.append(i[2])
    knot.append(i[3])

  mean  = np.mean(filament),np.mean(sheet),np.mean(void),np.mean(knot)
  std   = np.std(filament),np.std(sheet),np.std(void),np.std(knot)

  return mean, std

In [7]:
def process_in_chunks_split(n_total, numero, chunk_size):
    dict_classes_data_total = {}
    dict_classes_rand_total = {}

    for start in range(0, n_total, chunk_size):
        end = min(start + chunk_size, n_total)
        print(f"Random {start} a {end-1}")

        data_chunk, rand_chunk = dict_data(n_random=(end-start), numero=numero, start_index=start)

        dict_classes_data_chunk = classes(data_chunk)
        dict_classes_rand_chunk = classes(rand_chunk)

        #for tid, class_list in dict_classes_data_chunk.items():
        #    dict_classes_data_total.setdefault(tid, []).extend(class_list)
        #for tid, class_list in dict_classes_rand_chunk.items():
        #   dict_classes_rand_total.setdefault(tid, []).extend(class_list)

        df_data_chunk = pd.DataFrame([
            {'TARGETID': tid, 'classes': class_list}
            for tid, class_list in dict_classes_data_chunk.items()
        ])

        filename_data = f"/content/drive/MyDrive/DESI/count_fraction/LRG_NGC_{numero}_chunk_{start}_{end-1}_data.parquet"
        df_data_chunk.to_parquet(filename_data, index=False)

        df_rand_chunk = pd.DataFrame([
            {'TARGETID': tid, 'classes': class_list}
            for tid, class_list in dict_classes_rand_chunk.items()
        ])

        filename_rand = f"/content/drive/MyDrive/DESI/count_fraction/LRG_NGC_{numero}_chunk_{start}_{end-1}_rand.parquet"
        df_rand_chunk.to_parquet(filename_rand, index=False)

        del data_chunk, rand_chunk, dict_classes_data_chunk, dict_classes_rand_chunk, df_data_chunk, df_rand_chunk
        gc.collect()

    return dict_classes_data_total, dict_classes_rand_total

    #class_fractions_data = compute_class_fractions(dict_classes_data_total,n_total)
    #class_fractions_rand = compute_class_fractions(dict_classes_rand_total,n_total)

    #mean_data, std_data = cf_total(class_fractions_data)
    #mean_rand, std_rand = cf_total(class_fractions_rand)

    #return (class_fractions_data, mean_data, std_data,class_fractions_rand, mean_rand, std_rand)

In [ ]:
%%time
classes_data, classes_rand = process_in_chunks_split(n_total=100, numero=1, chunk_size=10)

Random 0 a 9
Random 10 a 19
Random 20 a 29
Random 30 a 39
Random 40 a 49
Random 50 a 59
Random 60 a 69
Random 70 a 79
Random 80 a 89
Random 90 a 99
CPU times: user 10min 40s, sys: 1min 1s, total: 11min 42s
Wall time: 13min


In [ ]:
%%time
classes_data2, classes_rand2 = process_in_chunks_split(n_total=100, numero=2, chunk_size=10)

Random 0 a 9
Random 10 a 19
Random 20 a 29
Random 30 a 39
Random 40 a 49
Random 50 a 59
Random 60 a 69
Random 70 a 79
Random 80 a 89
Random 90 a 99
CPU times: user 7min 42s, sys: 34.5 s, total: 8min 16s
Wall time: 9min 49s


In [8]:
def process_in_chunks_class_fracctions_data(n_total, numero, chunk_size):

    dict_classes_data_total = defaultdict(list)
    data_files = sorted(glob.glob(f"/content/drive/MyDrive/DESI/count_fraction/LRG_NGC_{numero}_chunk_*_data.parquet"))

    for fdata in data_files:
        print(f"Procesando {fdata}")

        df_data_chunk = pd.read_parquet(fdata)

        for _, row in df_data_chunk.iterrows():
            for c in row['classes']:
                dict_classes_data_total[row['TARGETID']].append(c)

        del df_data_chunk
        gc.collect()

    class_fractions_data = compute_class_fractions(dict_classes_data_total,n_total)
    df_class_fractions_data = pd.DataFrame([
        {
            'TARGETID': tid,
            'filament': vals[0],
            'sheet': vals[1],
            'void': vals[2],
            'knot': vals[3]
        }
        for tid, vals in class_fractions_data.items()
    ])

    mean_data, std_data = cf_total(class_fractions_data)

    #filename_data = f"/content/drive/MyDrive/DESI/count_fraction/LRG_NGC_{numero}_data_countfraction.parquet"
    #df_class_fractions_data.to_parquet(filename_data, index=False)
    #mean_data, std_data = cf_total(class_fractions_data)
    #mean_rand, std_rand = cf_total(class_fractions_rand)

    return mean_data,std_data
    #return (class_fractions_data, mean_data, std_data,class_fractions_rand, mean_rand, std_rand)

In [12]:
def process_in_chunks_class_fracctions_rand(n_total, numero, chunk_size):
    filament = 0
    sheet = 0
    void = 0
    knot = 0

    rand_files = sorted(glob.glob(
        f"/content/drive/MyDrive/DESI/count_fraction/LRG_NGC_{numero}_chunk_*_rand.parquet"
    ))
    print("Files:", len(rand_files))

    for frand in rand_files:
        print(f"Rand: {frand}")
        df_rand_chunk = pd.read_parquet(frand)

        clases = df_rand_chunk['classes'].apply(
            lambda x: str(x[0]) if hasattr(x, '__getitem__') and len(x) > 0 else None
        )

        counts = clases.value_counts()

        filament += counts.get('filament', 0)
        sheet += counts.get('sheet', 0)
        void += counts.get('void', 0)
        knot += counts.get('knot', 0)

        del df_rand_chunk
        gc.collect()

    print("Filament total:", filament)
    print("Sheet total:", sheet)
    print("Void total:", void)
    print("Knot total:", knot)

    n_total = filament + sheet + void + knot

    if n_total == 0:
        raise ValueError("n_total no puede ser 0")

    count_filament = (filament / n_total) * 100
    count_sheet = (sheet / n_total) * 100
    count_void = (void / n_total) * 100
    count_knot = (knot / n_total) * 100

    return count_filament, count_sheet, count_void, count_knot

## NGC - 1

In [12]:
%%time
mean_data,std_data = process_in_chunks_class_fracctions_data(n_total=100, numero=1,chunk_size=10)

Procesando /content/drive/MyDrive/DESI/count_fraction/LRG_NGC_1_chunk_0_9_data.parquet
Procesando /content/drive/MyDrive/DESI/count_fraction/LRG_NGC_1_chunk_10_19_data.parquet
Procesando /content/drive/MyDrive/DESI/count_fraction/LRG_NGC_1_chunk_20_29_data.parquet
Procesando /content/drive/MyDrive/DESI/count_fraction/LRG_NGC_1_chunk_30_39_data.parquet
Procesando /content/drive/MyDrive/DESI/count_fraction/LRG_NGC_1_chunk_40_49_data.parquet
Procesando /content/drive/MyDrive/DESI/count_fraction/LRG_NGC_1_chunk_50_59_data.parquet
Procesando /content/drive/MyDrive/DESI/count_fraction/LRG_NGC_1_chunk_60_69_data.parquet
Procesando /content/drive/MyDrive/DESI/count_fraction/LRG_NGC_1_chunk_70_79_data.parquet
Procesando /content/drive/MyDrive/DESI/count_fraction/LRG_NGC_1_chunk_80_89_data.parquet
Procesando /content/drive/MyDrive/DESI/count_fraction/LRG_NGC_1_chunk_90_99_data.parquet
CPU times: user 9min 39s, sys: 8.75 s, total: 9min 48s
Wall time: 10min 4s


In [27]:
columnas = ["Filament (%)", "Sheet (%)", "Void (%)", "Knot (%)"]
print('NGC 1')
print("Datos reales:")
for name, m, s in zip(columnas, mean_data, std_data):
    print(f"{name}: {m:.2f} ± {s:.2f}")

NGC 1
Datos reales:
Filament (%): 54.11 ± 36.02
Sheet (%): 44.53 ± 36.84
Void (%): 0.14 ± 1.49
Knot (%): 1.22 ± 5.93


In [13]:
%%time
count_filament, count_sheet, count_void, count_knot = process_in_chunks_class_fracctions_rand(n_total=100, numero=1,chunk_size=10)

Files: 10
Rand: /content/drive/MyDrive/DESI/count_fraction/LRG_NGC_1_chunk_0_9_rand.parquet
Rand: /content/drive/MyDrive/DESI/count_fraction/LRG_NGC_1_chunk_10_19_rand.parquet
Rand: /content/drive/MyDrive/DESI/count_fraction/LRG_NGC_1_chunk_20_29_rand.parquet
Rand: /content/drive/MyDrive/DESI/count_fraction/LRG_NGC_1_chunk_30_39_rand.parquet
Rand: /content/drive/MyDrive/DESI/count_fraction/LRG_NGC_1_chunk_40_49_rand.parquet
Rand: /content/drive/MyDrive/DESI/count_fraction/LRG_NGC_1_chunk_50_59_rand.parquet
Rand: /content/drive/MyDrive/DESI/count_fraction/LRG_NGC_1_chunk_60_69_rand.parquet
Rand: /content/drive/MyDrive/DESI/count_fraction/LRG_NGC_1_chunk_70_79_rand.parquet
Rand: /content/drive/MyDrive/DESI/count_fraction/LRG_NGC_1_chunk_80_89_rand.parquet
Rand: /content/drive/MyDrive/DESI/count_fraction/LRG_NGC_1_chunk_90_99_rand.parquet
Filament total: 18839654
Sheet total: 49951286
Void total: 717647
Knot total: 49230
CPU times: user 59.2 s, sys: 6.23 s, total: 1min 5s
Wall time: 1min 

In [14]:
columnas = ["Filament (%)", "Sheet (%)", "Void (%)", "Knot (%)"]
count = [count_filament, count_sheet, count_void, count_knot]
print('NGC 1')
print("Datos aleatorios:")
for name, count in zip(columnas, count):
  print(f'{name}: {count:.2f}')

NGC 1
Datos aleatorios:
Filament (%): 27.08
Sheet (%): 71.81
Void (%): 1.03
Knot (%): 0.07


##NGC - 2

In [28]:
%%time
mean_data2,std_data2 = process_in_chunks_class_fracctions_data(n_total=100, numero=2, chunk_size=10)

Procesando /content/drive/MyDrive/DESI/count_fraction/LRG_NGC_2_chunk_0_9_data.parquet
Procesando /content/drive/MyDrive/DESI/count_fraction/LRG_NGC_2_chunk_10_19_data.parquet
Procesando /content/drive/MyDrive/DESI/count_fraction/LRG_NGC_2_chunk_20_29_data.parquet
Procesando /content/drive/MyDrive/DESI/count_fraction/LRG_NGC_2_chunk_30_39_data.parquet
Procesando /content/drive/MyDrive/DESI/count_fraction/LRG_NGC_2_chunk_40_49_data.parquet
Procesando /content/drive/MyDrive/DESI/count_fraction/LRG_NGC_2_chunk_50_59_data.parquet
Procesando /content/drive/MyDrive/DESI/count_fraction/LRG_NGC_2_chunk_60_69_data.parquet
Procesando /content/drive/MyDrive/DESI/count_fraction/LRG_NGC_2_chunk_70_79_data.parquet
Procesando /content/drive/MyDrive/DESI/count_fraction/LRG_NGC_2_chunk_80_89_data.parquet
Procesando /content/drive/MyDrive/DESI/count_fraction/LRG_NGC_2_chunk_90_99_data.parquet
CPU times: user 2min 54s, sys: 1.02 s, total: 2min 55s
Wall time: 3min 19s


In [29]:
print('NGC 2')
print("Datos reales:")
for name, m, s in zip(columnas, mean_data2, std_data2):
    print(f"{name}: {m:.2f} ± {s:.2f}")

NGC 2
Datos reales:
Filament (%): 13.55 ± 26.09
Sheet (%): 82.71 ± 27.39
Void (%): 3.64 ± 12.41
Knot (%): 0.10 ± 1.47


In [16]:
%%time
count_filament2, count_sheet2, count_void2, count_knot2 = process_in_chunks_class_fracctions_rand(n_total=100, numero=2, chunk_size=10)

Files: 10
Rand: /content/drive/MyDrive/DESI/count_fraction/LRG_NGC_2_chunk_0_9_rand.parquet
Rand: /content/drive/MyDrive/DESI/count_fraction/LRG_NGC_2_chunk_10_19_rand.parquet
Rand: /content/drive/MyDrive/DESI/count_fraction/LRG_NGC_2_chunk_20_29_rand.parquet
Rand: /content/drive/MyDrive/DESI/count_fraction/LRG_NGC_2_chunk_30_39_rand.parquet
Rand: /content/drive/MyDrive/DESI/count_fraction/LRG_NGC_2_chunk_40_49_rand.parquet
Rand: /content/drive/MyDrive/DESI/count_fraction/LRG_NGC_2_chunk_50_59_rand.parquet
Rand: /content/drive/MyDrive/DESI/count_fraction/LRG_NGC_2_chunk_60_69_rand.parquet
Rand: /content/drive/MyDrive/DESI/count_fraction/LRG_NGC_2_chunk_70_79_rand.parquet
Rand: /content/drive/MyDrive/DESI/count_fraction/LRG_NGC_2_chunk_80_89_rand.parquet
Rand: /content/drive/MyDrive/DESI/count_fraction/LRG_NGC_2_chunk_90_99_rand.parquet
Filament total: 1311517
Sheet total: 51351368
Void total: 10059458
Knot total: 1729
CPU times: user 56.5 s, sys: 6.73 s, total: 1min 3s
Wall time: 1min 

In [17]:
columnas = ["Filament (%)", "Sheet (%)", "Void (%)", "Knot (%)"]
count = [count_filament2, count_sheet2, count_void2, count_knot2]
print('NGC 2')
print("Datos aleatorios:")
for name, count in zip(columnas, count):
  print(f'{name}: {count:.2f}')

NGC 2
Datos aleatorios:
Filament (%): 2.09
Sheet (%): 81.87
Void (%): 16.04
Knot (%): 0.00
